# Global NorESM — Per-Level Susceptibility Pipeline & CDNC/S_max Analysis

Companion to `noresm_global_susc_clustering`. Two parts:

1. **Per-level susceptibility compute pipeline** — the joblib-parallel regression that
   produces the `susceptibility_lev*.nc` files the clustering notebook later reads.
2. **CDNC / S_max / updraft analysis** — relationships between cloud-base CDNC, the
   S_max ratio diagnostics, and sub-grid updraft.

## 1. Per-level susceptibility pipeline

### 1.1 Setup

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from scipy.special import erfcinv
from pathlib import Path
from joblib import Parallel, delayed
import warnings


In [2]:
# Combined NorESM file: CCN(time, lev, radius, lat, lon) + CDNC(time, lev, lat, lon)
NorPath = '/share/pech2273/combined.nc'
ds = xr.open_dataset(NorPath, chunks={})


### 1.2 Regression configuration and helpers

`compute_cell_no_updraft` regresses log CDNC on log CCN for one grid cell using
equal-count log-CCN bins through their **medians** (the same convention as the station
notebooks). `process_lat_row_no_updraft` runs a full latitude row; the per-level loop
parallelizes rows with joblib. No updraft splitting here — `Total` only.

In [3]:
N_BINS         = 100
MIN_BIN_POINTS = 2
MIN_CELL_POINTS = 10   # skip grid cells with fewer valid CDNC samples than this
BINNING        = 'Equal Number'   # equal-count log-CCN bins (vs 'Equal Width')

# 16 workers on abakus's 32 cores: leaves headroom and avoids memory pressure
# from large per-level arrays being shared across many workers.
N_JOBS = 16


In [4]:
def compute_cell_no_updraft(CCN_data, CDNC_data, n_radii):
    """
    CCN_data:  (time, radius)
    CDNC_data: (time,)
    Regress log10(CDNC) on log10(CCN) per radius using equal-count log-CCN bins
    through their MEDIAN (consistent with the station notebooks). No updraft split.

    Vectorized: bin edges from quantiles, medians via scipy.stats.binned_statistic
    (C-level), replacing the per-bin Python masking loop. ~2.4x faster.
    """
    out = {k: np.full(n_radii, np.nan)
           for k in ['slope', 'r_value', 'std_err', 'intercept']}

    finite_cdnc = np.isfinite(CDNC_data) & (CDNC_data > 0)

    for r_idx in range(n_radii):
        CCN_r = CCN_data[:, r_idx]
        v = finite_cdnc & np.isfinite(CCN_r) & (CCN_r > 0)
        npts = int(v.sum())
        if npts < 2:
            continue

        lx = np.log10(CCN_r[v])
        ly = np.log10(CDNC_data[v])

        # Equal-count bins: quantile edges (can't have more bins than points).
        n_bins = min(N_BINS, npts)
        edges = np.unique(np.quantile(lx, np.linspace(0, 1, n_bins + 1)))
        if len(edges) < 3:
            continue

        # Per-bin medians in C (binned_statistic), not a Python loop.
        lx_med, _, _ = stats.binned_statistic(lx, lx, statistic='median', bins=edges)
        ly_med, _, _ = stats.binned_statistic(lx, ly, statistic='median', bins=edges)

        finite = np.isfinite(lx_med) & np.isfinite(ly_med)
        if finite.sum() < MIN_BIN_POINTS:
            continue

        slope, intercept, r_value, _, std_err = stats.linregress(
            lx_med[finite], ly_med[finite]
        )
        out['slope'][r_idx]     = slope
        out['r_value'][r_idx]   = r_value
        out['std_err'][r_idx]   = std_err
        out['intercept'][r_idx] = intercept

    return out


def process_lat_row_no_updraft(i, CCN_lev, CDNC_lev, n_lons, n_radii):
    """Process all longitude points for a single latitude row.
    Cells with fewer than MIN_CELL_POINTS valid CDNC samples are skipped
    (left NaN) before entering the per-radius regression — most ocean/clear-sky
    columns have almost no cloud-base CDNC, so this avoids wasted work."""
    row = {k: np.full((n_lons, n_radii), np.nan)
           for k in ['slope', 'r_value', 'std_err', 'intercept']}

    for j in range(n_lons):
        CDNC_data = CDNC_lev[:, i, j]     # (time,)

        # Cheap pre-filter: skip dead cells before the 21-radius loop.
        if np.count_nonzero(np.isfinite(CDNC_data) & (CDNC_data > 0)) < MIN_CELL_POINTS:
            continue

        CCN_data = CCN_lev[:, :, i, j]   # (time, radius)
        res = compute_cell_no_updraft(CCN_data, CDNC_data, n_radii)
        for k in row:
            row[k][j] = res[k]

    return i, row


def make_output_dataset_no_updraft(lats, lons, radii):
    shape = (len(lats), len(lons), len(radii))
    return xr.Dataset(
        data_vars={
            'slope':     (['lat', 'lon', 'radius'], np.full(shape, np.nan)),
            'r_value':   (['lat', 'lon', 'radius'], np.full(shape, np.nan)),
            'std_err':   (['lat', 'lon', 'radius'], np.full(shape, np.nan)),
            'intercept': (['lat', 'lon', 'radius'], np.full(shape, np.nan)),
        },
        coords={'lat': lats, 'lon': lons, 'radius': radii}
    )

In [5]:
# Grid metadata
radii   = ds.radius.values
lats    = ds.lat.values
lons    = ds.lon.values
levs    = ds.lev.values
n_radii = len(radii)
n_lats  = len(lats)
n_lons  = len(lons)
print(f'Grid: {n_lats} lat x {n_lons} lon, {len(levs)} levels, {n_radii} radii')


Grid: 96 lat x 144 lon, 10 levels, 21 radii


In [6]:
# Silence the expected log10(0)/subtract warnings from empty/degenerate cells.
warnings.filterwarnings('ignore', message='divide by zero encountered in log10')
warnings.filterwarnings('ignore', message='invalid value encountered in subtract')


### 1.3 Run the per-level regression

Loops levels, loads each into memory, regresses every (lat, lon) cell in parallel, and
collects a `(lat, lon, radius)` susceptibility dataset per level. **Heavy** — loads a
full level into RAM at a time.

In [7]:
results_per_lev = {}

for lev_idx, lev in enumerate(levs):
    print(f'\nProcessing level {lev:.1f} hPa ({lev_idx+1}/{len(levs)})...')

    # float32 halves memory bandwidth; precision is ample for a regression slope.
    CCN_lev  = ds['CCN'].isel(lev=lev_idx).transpose('time', 'radius', 'lat', 'lon').load().values.astype(np.float32)
    CDNC_lev = ds['CDNC'].isel(lev=lev_idx).transpose('time', 'lat', 'lon').load().values.astype(np.float32)

    results = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(process_lat_row_no_updraft)(i, CCN_lev, CDNC_lev, n_lons, n_radii)
        for i in range(n_lats)
    )

    ds_out = make_output_dataset_no_updraft(lats, lons, radii)
    for i, row in results:
        for k in ['slope', 'r_value', 'std_err', 'intercept']:
            ds_out[k].values[i] = row[k]

    results_per_lev[f'{lev:.1f}hPa'] = ds_out
    print(f'Done: {lev:.1f} hPa')

print('\nAll levels done!')



Processing level 691.4 hPa (1/10)...


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed:   31.9s
[Parallel(n_jobs=16)]: Done  85 out of  96 | elapsed:   56.3s remaining:    7.3s
[Parallel(n_jobs=16)]: Done  96 out of  96 | elapsed:  1.0min finished


Done: 691.4 hPa

Processing level 763.4 hPa (2/10)...


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed:   29.2s
[Parallel(n_jobs=16)]: Done  85 out of  96 | elapsed:   53.2s remaining:    6.9s
[Parallel(n_jobs=16)]: Done  96 out of  96 | elapsed:   58.1s finished


Done: 763.4 hPa

Processing level 820.9 hPa (3/10)...


...

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed:   23.3s
[Parallel(n_jobs=16)]: Done  85 out of  96 | elapsed:   39.3s remaining:    5.1s
[Parallel(n_jobs=16)]: Done  96 out of  96 | elapsed:   43.1s finished


Done: 976.3 hPa

Processing level 992.6 hPa (10/10)...


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed:   21.4s
[Parallel(n_jobs=16)]: Done  85 out of  96 | elapsed:   38.2s remaining:    4.9s


Done: 992.6 hPa

All levels done!


[Parallel(n_jobs=16)]: Done  96 out of  96 | elapsed:   43.4s finished


### 1.4 Save per-level results

Writes one file per level. These are later combined into
`susceptibility_all_levels.nc` for the clustering notebook.

In [8]:
SUSC_DIR = Path('/share/pech2273/susceptibility')
SUSC_DIR.mkdir(parents=True, exist_ok=True)

for lev_label, ds_lev in results_per_lev.items():
    out = SUSC_DIR / f'susceptibility_lev{lev_label}.nc'
    ds_lev.to_netcdf(out)
    print('wrote', out)


wrote /share/pech2273/susceptibility/susceptibility_lev691.4hPa.nc
wrote /share/pech2273/susceptibility/susceptibility_lev763.4hPa.nc
wrote /share/pech2273/susceptibility/susceptibility_lev820.9hPa.nc
wrote /share/pech2273/susceptibility/susceptibility_lev859.5hPa.nc
wrote /share/pech2273/susceptibility/susceptibility_lev887.0hPa.nc
wrote /share/pech2273/susceptibility/susceptibility_lev912.6hPa.nc
wrote /share/pech2273/susceptibility/susceptibility_lev936.2hPa.nc
wrote /share/pech2273/susceptibility/susceptibility_lev957.5hPa.nc
wrote /share/pech2273/susceptibility/susceptibility_lev976.3hPa.nc
wrote /share/pech2273/susceptibility/susceptibility_lev992.6hPa.nc


# 2 Level Analysis 

## 2.1 Visualize the Levels
In this section we will plot the levels maps of susceptibility 

In [2]:
results_per_lev

NameError: name 'results_per_lev' is not defined